In [1]:
import copy
import functools
import glob
import math
import pickle
import scipy
import sklearn
import warnings

warnings.filterwarnings('ignore')

import itertools
import graphviz as gr
import numpy as np
import os
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import seaborn as sns


from matplotlib import style
from matplotlib import pyplot as plt
from pprint import pprint
from scipy import stats, special
from sklearn import datasets, mixture
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from scipy.stats import ttest_ind


from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

import plotly.express as px
import plotly.figure_factory as ff

%matplotlib inline

style.use("fivethirtyeight")
pd.set_option("display.max_columns", 6)

np.random.seed(0)
pd.set_option('display.max_columns', None)

import dask
import dask.dataframe as dd
from dask.distributed import Client


### Import Historical Data and Census Data

In [2]:
dp_df = pd.read_csv("../data/CDPHE_TLGRF_historical.csv")

In [3]:
dp_df

,fips,county_x,state_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked
0,8031.0,Denver,Colorado,2020-03-18,0.253478,3.306363,4.857595,27.285714,128.714286,6.916333,101.428571,1.0,1.0,0.0,0.0,1.0,1.0
1,8037.0,Eagle,Colorado,2020-03-18,0.246840,3.235873,4.406719,25.428571,82.000000,6.276791,56.571429,1.0,2.0,0.0,0.0,2.0,2.0
2,8031.0,Denver,Colorado,2020-03-19,0.267161,3.547151,5.058972,34.714286,157.428571,9.274309,122.714286,1.0,1.0,0.0,0.0,1.0,1.0
3,8037.0,Eagle,Colorado,2020-03-19,0.253065,3.438585,4.561368,31.142857,95.714286,7.881172,64.571429,1.0,2.0,0.0,0.0,2.0,2.0
4,8031.0,Denver,Colorado,2020-03-20,0.250975,3.774402,5.250776,43.571429,190.714286,10.935347,147.142857,1.0,1.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,8029.0,Delta,Colorado,2022-12-23,-0.017627,4.460970,4.252569,86.571429,70.285714,-1.525969,-16.285714,1.0,19.0,0.0,0.0,11.0,12.0
3236,8081.0,Moffat,Colorado,2022-12-23,-0.035139,3.809832,3.509411,45.142857,33.428571,-1.586254,-11.714286,1.0,20.0,0.0,0.0,12.0,11.0
3237,8067.0,La Plata,Colorado,2022-12-23,-0.016677,5.125663,4.924143,168.285714,137.571429,-2.806479,-30.714286,1.0,26.0,0.0,0.0,13.0,14.0
3238,8083.0,Montezuma,Colorado,2022-12-23,-0.029646,4.751124,4.550865,115.714286,94.714286,-3.430459,-21.000000,1.0,28.0,1.0,1.0,14.0,13.0


### Exclude dates where there is not enough TLGRF information to make decision

In [4]:
CDPHE_decisions = dp_df[dp_df["changepoint"] == 1]
DP_dates = sorted(CDPHE_decisions["date.x"].unique())
TLGRF_decisions = dp_df[dp_df["date.x"].isin(DP_dates)]
TLGRF_decisions = TLGRF_decisions[TLGRF_decisions["rank_filtered"] <= TLGRF_decisions["capacity"]]
CDPHE_decision_count = pd.DataFrame(CDPHE_decisions.groupby("date.x")["changepoint"].count())
TLGRF_decision_count = pd.DataFrame(TLGRF_decisions.groupby("date.x")["changepoint"].count())

# Obtain dates where TLGRF total != CDPHE total
decision_counts = pd.merge(CDPHE_decision_count, TLGRF_decision_count, on="date.x")
decision_counts = decision_counts.rename(columns={"changepoint_x":"CDPHE_counts", "changepoint_y":"TLGRF_counts"})
excluded_dates = decision_counts[decision_counts["CDPHE_counts"] != decision_counts["TLGRF_counts"]].index
filtered_DP_dates = sorted(list(set(DP_dates) - set(excluded_dates)))
#DP_dates

### CDPHE's and TLGRF's Decisions 

In [5]:
filtered_dp_df = dp_df[dp_df["date.x"].isin(filtered_DP_dates)]
#filtered_dp_df = dp_df
TLGRF_decision_mask = filtered_dp_df["rank_filtered"] <= filtered_dp_df["capacity"]
filtered_dp_df["TLGRF_decision"] = TLGRF_decision_mask.astype(int)
filtered_dp_df["CDPHE_decision"] = filtered_dp_df["changepoint"].astype(int)
filtered_dp_df

,fips,county_x,state_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision
10,8031.0,Denver,Colorado,2020-03-25,0.214118,4.857595,6.035481,128.714286,418.000000,27.560029,289.285714,1.0,1.0,1.0,1.0,1.0,1.0,1,1
11,8041.0,El Paso,Colorado,2020-03-25,0.270288,4.110874,5.390376,61.000000,219.285714,16.487597,158.285714,1.0,2.0,0.0,0.0,2.0,2.0,0,0
12,8037.0,Eagle,Colorado,2020-03-25,0.190914,4.406719,5.307560,82.000000,201.857143,15.654948,119.857143,1.0,3.0,0.0,0.0,3.0,3.0,0,0
13,8035.0,Douglas,Colorado,2020-03-25,0.218179,3.575551,4.670155,35.714286,106.714286,7.792108,71.000000,1.0,7.0,0.0,0.0,4.0,4.0,0,0
14,8051.0,Gunnison,Colorado,2020-03-25,0.239097,3.279837,4.317488,26.571429,75.000000,6.353157,48.428571,1.0,9.0,0.0,0.0,5.0,5.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,8029.0,Delta,Colorado,2022-12-23,-0.017627,4.460970,4.252569,86.571429,70.285714,-1.525969,-16.285714,1.0,19.0,0.0,0.0,11.0,12.0,0,0
3236,8081.0,Moffat,Colorado,2022-12-23,-0.035139,3.809832,3.509411,45.142857,33.428571,-1.586254,-11.714286,1.0,20.0,0.0,0.0,12.0,11.0,0,0
3237,8067.0,La Plata,Colorado,2022-12-23,-0.016677,5.125663,4.924143,168.285714,137.571429,-2.806479,-30.714286,1.0,26.0,0.0,0.0,13.0,14.0,0,0
3238,8083.0,Montezuma,Colorado,2022-12-23,-0.029646,4.751124,4.550865,115.714286,94.714286,-3.430459,-21.000000,1.0,28.0,1.0,1.0,14.0,13.0,0,1


In [6]:
CDPHE_decisions = filtered_dp_df[filtered_dp_df["CDPHE_decision"] == 1]
TLGRF_decisions = filtered_dp_df[filtered_dp_df["TLGRF_decision"] == 1]

CDPHE_decision_count = pd.DataFrame(CDPHE_decisions.groupby("date.x")["changepoint"].count())
TLGRF_decision_count = pd.DataFrame(TLGRF_decisions.groupby("date.x")["changepoint"].count())
CDPHE_decision_count["changepoint"].agg(["sum","count"]), TLGRF_decision_count["changepoint"].agg(["sum","count"])

(sum      165
 count    145
 Name: changepoint, dtype: int64,
 sum      165
 count    145
 Name: changepoint, dtype: int64)

In [7]:
filtered_dp_df_old = filtered_dp_df[filtered_dp_df["date.x"] <= "2021-09-01"]
CDPHE_decisions_old = filtered_dp_df_old[filtered_dp_df_old["CDPHE_decision"] == 1]
TLGRF_decisions_old = filtered_dp_df_old[filtered_dp_df_old["TLGRF_decision"] == 1]

CDPHE_decision_count_old = pd.DataFrame(CDPHE_decisions_old.groupby("date.x")["changepoint"].count())
TLGRF_decision_count_old = pd.DataFrame(TLGRF_decisions_old.groupby("date.x")["changepoint"].count())
CDPHE_decision_count_old["changepoint"].agg(["sum","count"]), TLGRF_decision_count_old["changepoint"].agg(["sum","count"])

filtered_dp_df_old

,fips,county_x,state_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision
10,8031.0,Denver,Colorado,2020-03-25,0.214118,4.857595,6.035481,128.714286,418.000000,27.560029,289.285714,1.0,1.0,1.0,1.0,1.0,1.0,1,1
11,8041.0,El Paso,Colorado,2020-03-25,0.270288,4.110874,5.390376,61.000000,219.285714,16.487597,158.285714,1.0,2.0,0.0,0.0,2.0,2.0,0,0
12,8037.0,Eagle,Colorado,2020-03-25,0.190914,4.406719,5.307560,82.000000,201.857143,15.654948,119.857143,1.0,3.0,0.0,0.0,3.0,3.0,0,0
13,8035.0,Douglas,Colorado,2020-03-25,0.218179,3.575551,4.670155,35.714286,106.714286,7.792108,71.000000,1.0,7.0,0.0,0.0,4.0,4.0,0,0
14,8051.0,Gunnison,Colorado,2020-03-25,0.239097,3.279837,4.317488,26.571429,75.000000,6.353157,48.428571,1.0,9.0,0.0,0.0,5.0,5.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,8091.0,Ouray,Colorado,2021-08-30,0.023396,3.362358,3.479040,28.857143,32.428571,0.675143,3.571429,2.0,39.0,0.0,0.0,21.0,19.0,0,0
1484,8055.0,Huerfano,Colorado,2021-08-30,0.021689,3.337294,3.301114,28.142857,27.142857,0.610394,-1.000000,2.0,40.0,0.0,0.0,22.0,20.0,0,0
1485,8003.0,Alamosa,Colorado,2021-08-30,-0.003395,4.178773,4.529523,65.285714,92.714286,-0.221655,27.428571,2.0,44.0,1.0,1.0,23.0,4.0,0,1
1486,8111.0,San Juan,Colorado,2021-08-30,-0.011466,3.526361,3.247047,34.000000,25.714286,-0.389857,-8.285714,2.0,45.0,0.0,0.0,24.0,24.0,0,0


In [34]:
#unfiltered_dp_df = dp_df[dp_df["date.x"].isin(DP_dates)]
unfiltered_CDPHE_decisions = dp_df[dp_df["changepoint"] == 1]
unfiltered_CDPHE_decisions = unfiltered_CDPHE_decisions[unfiltered_CDPHE_decisions["date.x"] < "2021-09-30s"]

unfiltered_dp_df["CDPHE_decision"] = unfiltered_dp_df["changepoint"]
unfiltered_dp_df["TLGRF_decision"] = 0

unfiltered_DP_dates = sorted(unfiltered_CDPHE_decisions["date.x"].unique())
unfiltered_TLGRF_decisions = unfiltered_dp_df[unfiltered_dp_df["date.x"].isin(unfiltered_DP_dates)]
unfiltered_TLGRF_decisions = unfiltered_TLGRF_decisions[unfiltered_TLGRF_decisions["rank_filtered"] <= unfiltered_TLGRF_decisions["capacity"]]
unfiltered_TLGRF_decisions["TLGRF_decision"] = 1
unfiltered_dp_df.update(unfiltered_TLGRF_decisions)

unfiltered_CDPHE_decision_count = pd.DataFrame(unfiltered_CDPHE_decisions.groupby("date.x")["changepoint"].count())
unfiltered_TLGRF_decision_count = pd.DataFrame(unfiltered_TLGRF_decisions.groupby("date.x")["changepoint"].count())
unfiltered_CDPHE_decision_count["changepoint"].agg(["sum","count"]), unfiltered_TLGRF_decision_count["changepoint"].agg(["sum","count"])

(sum      99
 count    83
 Name: changepoint, dtype: int64,
 sum      107
 count     83
 Name: changepoint, dtype: int64)

In [9]:
unfiltered_dp_df

,fips,county_x,state_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,CDPHE_decision,TLGRF_decision
6,8031.0,Denver,Colorado,2020-03-23,0.228447,4.446007,5.749393,85.285714,314.000000,19.483248,228.714286,2.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0
7,8037.0,Eagle,Colorado,2020-03-23,0.198125,4.120198,5.068005,61.571429,158.857143,12.198821,97.285714,2.0,2.0,0.0,0.0,2.0,4.0,0.0,1.0
8,8041.0,El Paso,Colorado,2020-03-23,0.270250,3.415382,5.114566,30.428571,166.428571,8.223314,136.000000,2.0,3.0,0.0,0.0,3.0,2.0,0.0,0.0
9,8059.0,Jefferson,Colorado,2020-03-23,0.201594,3.630039,5.056246,37.714286,157.000000,7.602956,119.285714,2.0,4.0,1.0,1.0,4.0,3.0,1.0,0.0
10,8031.0,Denver,Colorado,2020-03-25,0.214118,4.857595,6.035481,128.714286,418.000000,27.560029,289.285714,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,8029.0,Delta,Colorado,2022-12-23,-0.017627,4.460970,4.252569,86.571429,70.285714,-1.525969,-16.285714,1.0,19.0,0.0,0.0,11.0,12.0,0.0,0.0
3236,8081.0,Moffat,Colorado,2022-12-23,-0.035139,3.809832,3.509411,45.142857,33.428571,-1.586254,-11.714286,1.0,20.0,0.0,0.0,12.0,11.0,0.0,0.0
3237,8067.0,La Plata,Colorado,2022-12-23,-0.016677,5.125663,4.924143,168.285714,137.571429,-2.806479,-30.714286,1.0,26.0,0.0,0.0,13.0,14.0,0.0,0.0
3238,8083.0,Montezuma,Colorado,2022-12-23,-0.029646,4.751124,4.550865,115.714286,94.714286,-3.430459,-21.000000,1.0,28.0,1.0,1.0,14.0,13.0,1.0,0.0


### Calculate Binary Classification Performance

In [10]:
def binary_classification_performance(df, outcome_col="rank_filtered", predictor_col="TLGRF_decision"):
    # Compute TP, TN, FP, FN
    selected_df = df[df[predictor_col]==1]
    
    Results_Thresh = pd.DataFrame()
    #Total
    Total_Decision_Set = df.groupby('date.x')[outcome_col].count()
    Results_Thresh["Total"] = Total_Decision_Set
    #P
    P = df.groupby('date.x')["capacity"].max()
    Results_Thresh["P"] = P
    #TP
    TP = selected_df.groupby('date.x').apply(lambda x: (x[outcome_col] <= len(x)).sum())
    Results_Thresh["TP"] = TP
    #FP = FN
    FP = selected_df.groupby('date.x').apply(lambda x: (x[outcome_col] > len(x)).sum())
    Results_Thresh["FP"] = FP
    Results_Thresh["FN"] = FP
    #N
    Results_Thresh["N"] = Results_Thresh["Total"] - Results_Thresh["P"]
    Results_Thresh["TN"] = Results_Thresh["N"] - Results_Thresh["FN"]
    #TN
    Results_Thresh = Results_Thresh[["Total", "P", "N", "TP", "FP", "FN", "TN"]]
    
    TP = Results_Thresh.sum()["TP"]
    FP = Results_Thresh.sum()["FP"]
    FN = Results_Thresh.sum()["FN"]
    TN = Results_Thresh.sum()["TN"]


    PPV = Results_Thresh.sum()["TP"]/(Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FP"])
    FPV = Results_Thresh.sum()["FP"]/(Results_Thresh.sum()["TN"] + Results_Thresh.sum()["FP"])
    TPR = Results_Thresh.sum()["TP"]/(Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FN"])
    TNR = Results_Thresh.sum()["TN"]/(Results_Thresh.sum()["TN"] + Results_Thresh.sum()["FP"])

    Precision = Results_Thresh.sum()["TP"] / (Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FP"])
    Recall = Results_Thresh.sum()["TP"] / (Results_Thresh.sum()["TP"] + Results_Thresh.sum()["FN"])
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # create a pandas dataframe from the confusion matrix values
    confusion_table = pd.DataFrame({'actual_positive': [1, 1, 0, 0], 'predicted_positive': [0, 1, 0, 1], 'count': [FN, TP, TN, FP]})

    # create a pivot table to represent the confusion matrix
    confusion_matrix = pd.pivot_table(confusion_table, values='count', index=['actual_positive'], columns=['predicted_positive'])

    # set the column and index names of the pivot table
    confusion_matrix.columns.name = 'Predicted Label'
    confusion_matrix.index.name = 'Actual Label'
    #confusion_matrix = confusion_matrix.T


    print("PPV={}, FPV={}\nTPR={}, TNR={}\nF1={}".format(PPV, FPV, TPR, TNR, F1))
    print(Results_Thresh.sum())
    confusion_matrix
    
    return Results_Thresh, (PPV, FPV, TPR, TNR, F1), confusion_matrix

In [11]:
binary_classification_performance(filtered_dp_df, outcome_col="delta_ranked", predictor_col="CDPHE_decision")

PPV=0.08484848484848485, FPV=0.07347931873479319
TPR=0.08484848484848485, TNR=0.9265206812652068
F1=0.08484848484848485
Total    2220.0
P         165.0
N        2055.0
TP         14.0
FP        151.0
FN        151.0
TN       1904.0
dtype: float64


(            Total    P     N  TP  FP  FN    TN
 date.x                                        
 2020-03-25      5  1.0   4.0   1   0   0   4.0
 2020-03-30      5  1.0   4.0   0   1   1   3.0
 2020-04-09      8  1.0   7.0   0   1   1   6.0
 2020-04-21      7  1.0   6.0   1   0   0   6.0
 2020-04-28      8  1.0   7.0   0   1   1   6.0
 ...           ...  ...   ...  ..  ..  ..   ...
 2022-12-01     17  1.0  16.0   0   1   1  15.0
 2022-12-09     17  1.0  16.0   1   0   0  16.0
 2022-12-12     16  1.0  15.0   0   1   1  14.0
 2022-12-16     17  1.0  16.0   0   1   1  15.0
 2022-12-23     15  1.0  14.0   0   1   1  13.0
 
 [145 rows x 7 columns],
 (0.08484848484848485,
  0.07347931873479319,
  0.08484848484848485,
  0.9265206812652068,
  0.08484848484848485),
 Predicted Label       0      1
 Actual Label                  
 0                1904.0  151.0
 1                 151.0   14.0)

In [12]:
binary_classification_performance(filtered_dp_df, outcome_col="delta_ranked", predictor_col="TLGRF_decision")

PPV=0.6909090909090909, FPV=0.024817518248175182
TPR=0.6909090909090909, TNR=0.9751824817518249
F1=0.6909090909090909
Total    2220.0
P         165.0
N        2055.0
TP        114.0
FP         51.0
FN         51.0
TN       2004.0
dtype: float64


(            Total    P     N  TP  FP  FN    TN
 date.x                                        
 2020-03-25      5  1.0   4.0   1   0   0   4.0
 2020-03-30      5  1.0   4.0   1   0   0   4.0
 2020-04-09      8  1.0   7.0   1   0   0   7.0
 2020-04-21      7  1.0   6.0   1   0   0   6.0
 2020-04-28      8  1.0   7.0   1   0   0   7.0
 ...           ...  ...   ...  ..  ..  ..   ...
 2022-12-01     17  1.0  16.0   1   0   0  16.0
 2022-12-09     17  1.0  16.0   0   1   1  15.0
 2022-12-12     16  1.0  15.0   0   1   1  14.0
 2022-12-16     17  1.0  16.0   0   1   1  15.0
 2022-12-23     15  1.0  14.0   0   1   1  13.0
 
 [145 rows x 7 columns],
 (0.6909090909090909,
  0.024817518248175182,
  0.6909090909090909,
  0.9751824817518249,
  0.6909090909090909),
 Predicted Label       0      1
 Actual Label                  
 0                2004.0   51.0
 1                  51.0  114.0)

### Validation for Old Manuscript

In [35]:
binary_classification_performance(unfiltered_dp_df[unfiltered_dp_df["date.x"] <= "2021-09-01"], outcome_col="delta_ranked", predictor_col="TLGRF_decision")

PPV=0.6989247311827957, FPV=0.03160270880361174
TPR=0.6989247311827957, TNR=0.9683972911963883
F1=0.6989247311827957
Total    979.0
P         93.0
N        886.0
TP        65.0
FP        28.0
FN        28.0
TN       858.0
dtype: float64


(            Total    P     N  TP  FP  FN    TN
 date.x                                        
 2020-03-23      4  2.0   2.0   1   1   1   1.0
 2020-03-25      5  1.0   4.0   1   0   0   4.0
 2020-03-30      5  1.0   4.0   1   0   0   4.0
 2020-04-09      8  1.0   7.0   1   0   0   7.0
 2020-04-21      7  1.0   6.0   1   0   0   6.0
 ...           ...  ...   ...  ..  ..  ..   ...
 2021-08-03     15  1.0  14.0   1   0   0  14.0
 2021-08-04     14  1.0  13.0   1   0   0  13.0
 2021-08-25     25  1.0  24.0   1   0   0  24.0
 2021-08-26     24  2.0  22.0   2   0   0  22.0
 2021-08-30     25  2.0  23.0   1   1   1  22.0
 
 [74 rows x 7 columns],
 (0.6989247311827957,
  0.03160270880361174,
  0.6989247311827957,
  0.9683972911963883,
  0.6989247311827957),
 Predicted Label      0     1
 Actual Label                
 0                858.0  28.0
 1                 28.0  65.0)

In [36]:
binary_classification_performance(unfiltered_dp_df[unfiltered_dp_df["date.x"] <= "2021-09-01"], outcome_col="delta_ranked", predictor_col="CDPHE_decision")

PPV=0.09090909090909091, FPV=0.09029345372460497
TPR=0.09090909090909091, TNR=0.909706546275395
F1=0.09090909090909091
Total    979.0
P         93.0
N        886.0
TP         8.0
FP        80.0
FN        80.0
TN       806.0
dtype: float64


(            Total    P     N  TP  FP  FN    TN
 date.x                                        
 2020-03-23      4  2.0   2.0   0   1   1   1.0
 2020-03-25      5  1.0   4.0   1   0   0   4.0
 2020-03-30      5  1.0   4.0   0   1   1   3.0
 2020-04-09      8  1.0   7.0   0   1   1   6.0
 2020-04-21      7  1.0   6.0   1   0   0   6.0
 ...           ...  ...   ...  ..  ..  ..   ...
 2021-08-03     15  1.0  14.0   0   1   1  13.0
 2021-08-04     14  1.0  13.0   0   1   1  12.0
 2021-08-25     25  1.0  24.0   0   1   1  23.0
 2021-08-26     24  2.0  22.0   0   2   2  20.0
 2021-08-30     25  2.0  23.0   0   2   2  21.0
 
 [74 rows x 7 columns],
 (0.09090909090909091,
  0.09029345372460497,
  0.09090909090909091,
  0.909706546275395,
  0.09090909090909091),
 Predicted Label      0     1
 Actual Label                
 0                806.0  80.0
 1                 80.0   8.0)

In [15]:
(1921+20+97+97, 2004+114+51+51)

(2135, 2220)

### Investigate Sources of Discrepancy

In [16]:
### Load in Combined Census Data
augmented_data_path = "../../data/augmented_us-counties-states_latest.csv"
#augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True)).compute()
augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True))
augmented_colorado_df = augmented_df[augmented_df["state"]=="Colorado"]
augmented_colorado_df = augmented_colorado_df.compute()

In [17]:
augmented_colorado_df = augmented_colorado_df.sort_values(by=["date","fips"])
augmented_colorado_df

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_NOHSDP,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_MOBILE,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_FIPS_Code,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Closed_day_cares,Reopen_day_cares,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,Stay_at_home_order'_issued_but_did_not_specifically_restrict_movement_of_the_general_public,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Closed_businesses_overnight,Began_to_reopen_businesses,Religious_gatherings_exempt_without_clear_social_distance_mandate*,Face_mask_mandate_in_public_spaces,Face_mask_mandate_x2,Face_mask_mandate_enforced_by_fines,Face_mask_mandate_enforced_by_criminal_charge/citation,No_legal_enforcement_of_face_mask_mandate,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Ended_face_mask_mandate_x2,Attempted_to_prevent_local_governments_from_implementing_face_mask_orders,Banned_local_mask_mandates,Liquor_stores_remained_open,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Firearm_sellers_remained_open,Cannabis_dispensaries_considered_essential_business,Closed_restaurants_except_take_out,Reopen_restaurants,Initially_reopen_restaurants_for_outdoor_dining_only,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Allowed_businesses_to_reopen_overnight,Began_to_reclose_bars,Closed_bars_(x2),Closed_movie_theaters_(x2),Closed_hair_salons/barber_shops_(x2),Closed_gyms_(x2),Closed_restaurants_(x2),Reopened_restaurants_(x2),Reopened_bars_(x2),Reopened_gyms_(x2),Reopened_hair_salons/barber_shops_(x2),Reopened_movie_theaters_(x2),Closed_bars_(x3),Closed_restaurants_(x3),Reopened_bars_(x3),Reopened_restaurants_(x3),Mandate_quarantine_for_those_entering_the_state_from_specific_settings,Mandate_quarantine_for_all_individuals_entering_the_state,Date_all_mandated_quarantines_ended,Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_People_who_are_Incarcerated,Vaccine_allocation_phase:_Residents_of_Homeless_Shelters,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers

In [18]:
decision_set_augmented = pd.merge(filtered_dp_df, augmented_colorado_df, left_on=["fips","date.x"], right_on=["fips","date"])
cols_to_drop = decision_set_augmented.columns[decision_set_augmented.nunique()==1]
reduced_decision_set_augmented = decision_set_augmented.drop(cols_to_drop, axis=1)

In [19]:
TLGRF_decision_augmented = reduced_decision_set_augmented[reduced_decision_set_augmented["TLGRF_decision"]==1]
CDPHE_decision_augmented = reduced_decision_set_augmented[reduced_decision_set_augmented["CDPHE_decision"]==1]

CDPHE_decision_augmented["CDPHE_TP"] = CDPHE_decision_augmented["delta_ranked"] <= CDPHE_decision_augmented["capacity"]
TLGRF_decision_augmented["TLGRF_TP"] = TLGRF_decision_augmented["delta_ranked"] <= TLGRF_decision_augmented["capacity"]

CDPHE_decision_augmented["CDPHE_FP"] = CDPHE_decision_augmented["delta_ranked"] > CDPHE_decision_augmented["capacity"]
TLGRF_decision_augmented["TLGRF_FP"] = TLGRF_decision_augmented["delta_ranked"] > TLGRF_decision_augmented["capacity"]


reduced_decision_set_augmented["CDPHE_TP"] = False
reduced_decision_set_augmented["TLGRF_TP"] = False
reduced_decision_set_augmented["CDPHE_FP"] = False
reduced_decision_set_augmented["TLGRF_FP"] = False


reduced_decision_set_augmented.update(CDPHE_decision_augmented)
reduced_decision_set_augmented.update(TLGRF_decision_augmented)
reduced_decision_set_augmented

,fips,county_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases_x,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision,date,county,cases,deaths,datetime,days_from_start,rolled_cases_y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Began_to_reopen_businesses,Face_mask_mandate_in_public_spaces,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Closed_restaurants_except_take_out,Reopen_restaurants,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Began_to_reclose_bars,Closed_bars_(x2),Reopened_bars_(x2),Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Date_adults_ages_80+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_75+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_65+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_60+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_55+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_50+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_45+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_40+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_30+_became_eligible_for_COVID-19_vaccination,Date_K-12_school_employees_became_eligible_for_COVID-19_vaccination,Date_grocery_store_workers_became_eligible_for_COVID-19_vaccination,Date_incarcerated_people_became_eligible_for_COVID-19_vaccination,Date_general_public_became_eligible_for_COVID-19_vaccination,First_overall_eviction_moratorium_start,First_overall_eviction_moratorium_end,Second_overall_eviction_moratorium_start,Second_overall_eviction_moratorium_end

In [20]:
reduced_decision_set_augmented[reduced_decision_set_augmented["CDPHE_TP"] == 1]

,fips,county_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases_x,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision,date,county,cases,deaths,datetime,days_from_start,rolled_cases_y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Began_to_reopen_businesses,Face_mask_mandate_in_public_spaces,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Closed_restaurants_except_take_out,Reopen_restaurants,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Began_to_reclose_bars,Closed_bars_(x2),Reopened_bars_(x2),Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Date_adults_ages_80+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_75+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_65+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_60+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_55+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_50+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_45+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_40+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_30+_became_eligible_for_COVID-19_vaccination,Date_K-12_school_employees_became_eligible_for_COVID-19_vaccination,Date_grocery_store_workers_became_eligible_for_COVID-19_vaccination,Date_incarcerated_people_became_eligible_for_COVID-19_vaccination,Date_general_public_became_eligible_for_COVID-19_vaccination,First_overall_eviction_moratorium_start,First_overall_eviction_moratorium_end,Second_overall_eviction_moratorium_start,Second_overall_eviction_moratorium_end

### CDPHE_TP but TLGRF_FP

In [21]:
CDPHE_TP_TLGRF_FP_mask = (reduced_decision_set_augmented["CDPHE_TP"] == True) & (reduced_decision_set_augmented["TLGRF_TP"] == False)
CDPHE_TP_TLGRF_FP = reduced_decision_set_augmented[CDPHE_TP_TLGRF_FP_mask]
CDPHE_TP_TLGRF_FP

,fips,county_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases_x,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision,date,county,cases,deaths,datetime,days_from_start,rolled_cases_y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Began_to_reopen_businesses,Face_mask_mandate_in_public_spaces,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Closed_restaurants_except_take_out,Reopen_restaurants,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Began_to_reclose_bars,Closed_bars_(x2),Reopened_bars_(x2),Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Date_adults_ages_80+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_75+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_65+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_60+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_55+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_50+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_45+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_40+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_30+_became_eligible_for_COVID-19_vaccination,Date_K-12_school_employees_became_eligible_for_COVID-19_vaccination,Date_grocery_store_workers_became_eligible_for_COVID-19_vaccination,Date_incarcerated_people_became_eligible_for_COVID-19_vaccination,Date_general_public_became_eligible_for_COVID-19_vaccination,First_overall_eviction_moratorium_start,First_overall_eviction_moratorium_end,Second_overall_eviction_moratorium_start,Second_overall_eviction_moratorium_end

### TLGRF_TP but CDPHE_FP

In [22]:
CDPHE_FP_TLGRF_TP_mask = (reduced_decision_set_augmented["CDPHE_TP"] == False) & (reduced_decision_set_augmented["TLGRF_TP"] == True)
CDPHE_FP_TLGRF_TP = reduced_decision_set_augmented[CDPHE_FP_TLGRF_TP_mask]
CDPHE_FP_TLGRF_TP

,fips,county_x,date.x,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases_x,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision,date,county,cases,deaths,datetime,days_from_start,rolled_cases_y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Began_to_reopen_businesses,Face_mask_mandate_in_public_spaces,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Closed_restaurants_except_take_out,Reopen_restaurants,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Began_to_reclose_bars,Closed_bars_(x2),Reopened_bars_(x2),Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Date_adults_ages_80+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_75+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_65+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_60+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_55+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_50+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_45+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_40+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_30+_became_eligible_for_COVID-19_vaccination,Date_K-12_school_employees_became_eligible_for_COVID-19_vaccination,Date_grocery_store_workers_became_eligible_for_COVID-19_vaccination,Date_incarcerated_people_became_eligible_for_COVID-19_vaccination,Date_general_public_became_eligible_for_COVID-19_vaccination,First_overall_eviction_moratorium_start,First_overall_eviction_moratorium_end,Second_overall_eviction_moratorium_start,Second_overall_eviction_moratorium_end

In [23]:
CDPHE_TP_TLGRF_FP.describe()

,fips,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases_x,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision,cases,deaths,days_from_start,rolled_cases_y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Began_to_reopen_businesses,Face_mask_mandate_in_public_spaces,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Closed_restaurants_except_take_out,Reopen_restaurants,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Began_to_reclose_bars,Closed_bars_(x2),Reopened_bars_(x2),Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Date_adults_ages_80+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_75+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_65+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_60+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_55+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_50+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_45+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_40+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_30+_became_eligible_for_COVID-19_vaccination,Date_K-12_school_employees_became_eligible_for_COVID-19_vaccination,Date_grocery_store_workers_became_eligible_for_COVID-19_vaccination,Date_incarcerated_people_became_eligible_for_COVID-19_vaccination,Date_general_public_became_eligible_for_COVID-19_vaccination,First_overall_eviction_moratorium_start,First_overall_eviction_moratorium_end,Second_overall_eviction_moratorium_start,Second_overall_eviction_moratorium_end,First_eviction_initiation_ban_start,

In [24]:
CDPHE_FP_TLGRF_TP.describe()

,fips,tau.hat,log_rolled_cases,shifted_log_rolled_cases,rolled_cases_x,shifted_rolled_cases,predictor,delta cases,capacity,rank_unfiltered,changepoint,outbreak,rank_filtered,delta_ranked,TLGRF_decision,CDPHE_decision,cases,deaths,days_from_start,rolled_cases_y,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Began_to_reopen_businesses,Face_mask_mandate_in_public_spaces,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Closed_restaurants_except_take_out,Reopen_restaurants,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Began_to_reclose_bars,Closed_bars_(x2),Reopened_bars_(x2),Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vaccine_allocation_phase:_Additional_Essential_Workers,Date_adults_ages_80+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_75+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_70+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_65+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_60+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_55+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_50+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_45+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_40+_became_eligible_for_COVID-19_vaccination,Date_adults_ages_30+_became_eligible_for_COVID-19_vaccination,Date_K-12_school_employees_became_eligible_for_COVID-19_vaccination,Date_grocery_store_workers_became_eligible_for_COVID-19_vaccination,Date_incarcerated_people_became_eligible_for_COVID-19_vaccination,Date_general_public_became_eligible_for_COVID-19_vaccination,First_overall_eviction_moratorium_start,First_overall_eviction_moratorium_end,Second_overall_eviction_moratorium_start,Second_overall_eviction_moratorium_end,First_eviction_initiation_ban_start,

### t-test

In [25]:
def t_test_results(df1, df2):
    LAT_loc = df1.columns.get_loc("LAT")
    CDPHE_TP_loc = df1.columns.get_loc("CDPHE_TP")
    t_test_results = pd.DataFrame(columns=['Column', 't-statistic', 'p-value', 'Standard Error', 'Confidence Interval'])

    for column in df1.columns[LAT_loc:CDPHE_TP_loc]:
        t_statistic, p_value = ttest_ind(df1[column], df2[column])
        mean1, mean2 = df1[column].mean(), df2[column].mean()
        std1, std2 = df1[column].std(), df2[column].std()
        n1, n2 = len(df1[column]), len(df2[column])

        standard_error = np.sqrt((std1 ** 2 / n1) + (std2 ** 2 / n2))
        confidence_interval = (mean1 - mean2) - (1.96 * standard_error), (mean1 - mean2) + (1.96 * standard_error)


        if not math.isnan(t_statistic) and not math.isinf(t_statistic):
            t_test_results = t_test_results.append({
            'Column': column,
            't-statistic': t_statistic,
            'p-value': p_value,
            'Standard Error': standard_error,
            'Confidence Interval': confidence_interval
        }, ignore_index=True)
            #print("Column:", column)
            #print("T-statistic:", t_statistic)
            #print("P-value:", p_value)
            #print()
    return t_test_results

In [26]:
CDPHE_right_vs_TLGRF_right = t_test_results(CDPHE_TP_TLGRF_FP, CDPHE_FP_TLGRF_TP)
CDPHE_right_vs_TLGRF_right = CDPHE_right_vs_TLGRF_right[CDPHE_right_vs_TLGRF_right["p-value"] <= 0.05].T
CDPHE_right_vs_TLGRF_right

,3,4,5,6,7,9,10,11,12,13,14,15,16,17,18,19,20,21,22,24,26,27,28,29,34,36,39,40,41,42,43,44,45,46,48,51,53,54,55,56,57,62,67,68,71,75,77,79,81,85,86,87,88,89,91,93,94,95,96
Column,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,MP_UNEMP,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_DISABL,MP_SNGPNT,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,MP_NOVEH,EPL_POV,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,RPL_THEME2,EPL_MUNIT,EPL_MOBILE,EPL_GROUPQ,RPL_THEMES,F_UNEMP,F_THEME1,F_AGE17,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_CROWD,F_THEME4,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP
t-statistic,-4.568835,-4.599507,-4.577662,-4.524236,-4.539199,-4.466744,-4.603176,-4.531312,-4.542365,-4.524077,-4.502784,-4.29883,-4.576323,-4.826527,-4.4792,-4.543534,-4.687868,2.7724,4.150551,3.604177,2.200115,4.636503,4.457232,2.552866,2.630845,3.882277,4.267731,4.174374,-2.506326,3.137885,3.902197,3.954351,2.926134,4.297591,4.014428,2.990615,3.980215,5.191577,2.972375,3.032959,3.394309,2.473738,-3.572746,3.921496,-3.243656,2.107614,3.933415,2.423941,4.820009,2.664333,4.820009,5.798464,-4.005697,2.664333,-3.657716,-4.477984,5.868925,3.843234,-4.57596
p-value,0.000013,0.000011,0.000012,0.000015,0.000014,0.000019,0.000011,0.000015,0.000014,0.000015,0.000017,0.000037,0.000012,0.000004,0.000018,0.000014,0.000008,0.006519,0.000065,0.000469,0.029852,0.00001,0.00002,0.01203,0.009717,0.000175,0.000041,0.000059,0.013635,0.002175,0.000163,0.000135,0.004156,0.000037,0.000108,0.003424,0.000123,0.000001,0.003618,0.00301,0.000952,0.014871,0.000522,0.000152,0.001556,0.037295,0.000146,0.016954,0.000005,0.008853,0.000005,0.0,0.000112,0.008853,0.000389,0.000018,0.0,0.000202,0.000012
Standard Error,28768.511358,11140.456062,10553.041671,3030.188191,858.698771,1263.411694,3438.972306,7135.393764,3388.496905,937.163399,8910.091427,547.018299,1447.374501,440.790417,328.166222,449.587727,757.044602,2.493501,0.441526,0.370939,846.525091,1.369738,0.188518,1.287812,0.190267,0.292126,0.88583,0.205483,3.280335,0.395486,2.157908,0.279767,0.547356,0.165548,0.291525,0.131819,0.126981,0.083636,0.432577,0.127367,0.099047,0.124925,0.115319,0.082065,0.131258,0.124754,0.184896,0.299399,0.184664,0.143163,0.184664,0.202458,0.148324,0.143163,0.187972,2292.372176,1.416416,0.271379,25585.915605
Confidence Interval,"(-557358.8256530971, -444586.2611292794)","(-216353.57292004646, -172682.98515739013)","(-205038.31280999738, -163670.38945969552)","(-58651.64816013898, -46773.310451342986)","(-16688.658402926794, -13322.559220571206)","(-22120.517894767097, -17167.94405449859)","(-66960.96649490841, -53480.195053823234)","(-136871.36376650207, -108900.62021213613)","(-65910.95460129385, -52628.04673381963)","(-18186.625307921375, -14512.944785536569)","(-169782.832600808, -134855.27420827077)","(-7972.306733915086, -5827.9950017324445)","(-27301.851352184993, -21628.143307361064)","(-6314.883797126655, -4586.985361751849)","(-5715.719812964098, -4429.308224419079)","(-8803.286738376355, -7040.902847738465)","(-15113.752679951986, -12146.137840742276)","(-1.5289177871921336, 8.245606705750214)","(0.9411504051880055, 2.6719337069615277)","(0.42716573536668745, 1.8812454795865858)","(-380.7472564005052, 2937.6311015273413)","(0.9533649519605074, 6.322736516664335)","(0.9058049242369142, 1.6447958768311792)","(0.8948466123534953, 5.943070610610464)","(0.5113222661319188, 1.2571690556304331)","(0.8366445712355645, 1.9817799948525532)","(-0.3439697531000496, 3.1284824366781594)","(0.3361050120257979, 1.141598592780611)","(-11.008094689703817, 1.8508183212124978)","(0.06583543555089477, 1.6161405324063818)","(2.8680979746363073, 11.327095616818951)","(0.9079981671336494, 2.0046854109704872)","(-0.40766367415339944, 1.7379707502548574)","(0.5796640070457182, 1.2286136965590877)","(0.4742722004513228, 1.617049561898479)","(-0.06822252508202414, 0.44850770532234546)","(0.04342163

In [27]:
CDPHE_TP = reduced_decision_set_augmented[reduced_decision_set_augmented["CDPHE_TP"] == 1]
TLGRF_FP = reduced_decision_set_augmented[reduced_decision_set_augmented["TLGRF_FP"] == 1]
CDPHE_TP_TLGRF_FP_t_test = t_test_results(CDPHE_TP, TLGRF_FP)
CDPHE_TP_TLGRF_FP_t_test = CDPHE_TP_TLGRF_FP_t_test[CDPHE_TP_TLGRF_FP_t_test["p-value"] <= 0.10]
CDPHE_TP_TLGRF_FP_t_test

,Column,t-statistic,p-value,Standard Error,Confidence Interval
17,E_MOBILE,-2.506026,0.014805,479.890634,"(-3102.764914975792, -1221.5936284415743)"
20,E_GROUPQ,-1.797327,0.077077,1516.809490,"(-6878.429794393585, -932.5365921610351)"
71,EPL_GROUPQ,-2.171059,0.033701,0.085730,"(-0.3659362430263178, -0.02987636201569885)"
76,F_UNEMP,1.975121,0.052640,0.099013,"(-0.07081663181013857, 0.31731523124991445)"
80,F_AGE17,3.671664,0.000498,0.113804,"(-0.008769987622679332, 0.4373414161941079)"
83,F_THEME2,2.159915,0.034592,0.140489,"(-0.035861776453116306, 0.5148533730917718)"
84,F_MINRTY,1.949966,0.055633,0.071429,"(-0.06857142857142856, 0.2114285714285714)"
178,metrics.testPositivityRatio,2.126852,0.037355,0.068831,"(-0.05194323597010839, 0.21787320795890394)"


In [28]:
CDPHE_FP = reduced_decision_set_augmented[reduced_decision_set_augmented["CDPHE_FP"] == 1]
TLGRF_TP = reduced_decision_set_augmented[reduced_decision_set_augmented["TLGRF_TP"] == 1]
CDPHE_FP_TLGRF_TP_t_test = t_test_results(CDPHE_FP, TLGRF_TP)
CDPHE_FP_TLGRF_TP_t_test = CDPHE_FP_TLGRF_TP_t_test[CDPHE_FP_TLGRF_TP_t_test["p-value"] <= 0.10]
CDPHE_FP_TLGRF_TP_t_test

,Column,t-statistic,p-value,Standard Error,Confidence Interval
1,LON,-1.919448,5.600958e-02,0.203413,"(-0.8051176568717431, -0.007737684362354613)"
2,AREA_SQMI,-4.104536,5.409961e-05,90.759062,"(-574.3992983023662, -218.62377442228225)"
3,E_TOTPOP,-18.407931,3.438089e-49,28735.231742,"(-521162.31975350034, -408520.21132585366)"
4,E_HU,-18.325152,6.705465e-49,11125.773691,"(-201172.17508478236, -157559.1422150898)"
5,E_HH,-18.364381,4.885678e-49,10597.014207,"(-191992.12443408612, -150451.82874355998)"
...,...,...,...,...,...
94,E_UNINSUR,-18.505654,1.563335e-49,2011.136082,"(-36558.7907610032, -28675.13732079069)"
95,EP_UNINSUR,6.791267,7.378981e-11,0.342909,"(1.845006671656144, 3.1892096638846965)"
96,MP_UNINSUR,11.075112,1.189632e-23,0.123979,"(1.1197813008746547, 1.6057792893426073)"
97,E_DAYPOP,-18.500794,1.625811e-49,25471.898679,"(-463753.6915388886, -363903.848719041)"


In [29]:
CDPHE_FP_TLGRF_FP_t_test = t_test_results(CDPHE_FP, TLGRF_FP)
CDPHE_FP_TLGRF_FP_t_test = CDPHE_FP_TLGRF_FP_t_test[CDPHE_FP_TLGRF_FP_t_test["p-value"] <= 0.10]
CDPHE_FP_TLGRF_FP_t_test

,Column,t-statistic,p-value,Standard Error,Confidence Interval
2,AREA_SQMI,-2.454025,1.498211e-02,164.327913,"(-697.6437924173192, -53.478371988962465)"
3,E_TOTPOP,-6.509432,5.951014e-10,41130.351346,"(-244673.1840033743, -83442.2067251025)"
4,E_HU,-6.473679,7.239669e-10,15863.341547,"(-94367.34265322104, -32183.043790098025)"
5,E_HH,-6.476045,7.146526e-10,15124.428634,"(-90011.14515291149, -30723.384908119544)"
6,E_POV,-6.523226,5.516572e-10,4368.733199,"(-25686.234666082222, -8560.800524152814)"
7,E_UNEMP,-6.696405,2.112545e-10,1236.926963,"(-7376.126750870245, -2527.373054349856)"
8,E_PCI,-1.868522,6.315163e-02,1441.372846,"(-5616.620839669088, 33.56071760701434)"
9,E_NOHSDP,-6.289641,1.964896e-09,1644.569722,"(-9663.845422081546, -3217.1321133034694)"
10,E_AGE65,-6.259051,2.315590e-09,4939.559735,"(-29010.576686616063, -9647.502523876085)"
11,E_AGE17,-6.575683,4.131361e-10,10143.745558,"(-60393.195649943984, -20629.71306321015)"


In [30]:
CDPHE_FP.shape

(151, 267)

In [31]:
TLGRF_FP.shape

(51, 267)